# [Experiment] Vanilla TabNet | Cosine Annealing LR | Higgs Boson

### Overview
This notebook evaluates the standard TabNet architecture under a shifted optimization paradigm, utilizing a Cosine Annealing learning rate schedule instead of the traditional StepLR.

### Methodological Context: Continuous Decay
While the structural configuration of the network remains rigorously identical to the original TabNet paper's setup, we are deliberately altering the "thermodynamic" optimization environment. We are departing from the original paper's StepLR schedule to evaluate how the unaltered architecture responds to a smooth, continuous decay, which contrasts sharply with sudden, discrete drops in learning rate.

### The Objective
The goal is to determine how the standard, historically validated architecture responds to a more exploratory and continuously adapting learning rate. This establishes a critical secondary baseline, allowing us to decouple the inherent capabilities of the original TabNet structure from the artifacts of the specific StepLR schedule used by the original authors.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'
df = pd.read_csv(url, header=None)

# Column 0 is the class label (1 for signal, 0 for background). Columns 1-28 are features.
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values.astype(int)

# Free up the massive raw DataFrame from memory
del df
gc.collect()

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"\nFinal Train shape: {X_train.shape}")
print(f"Final Valid shape: {X_valid.shape}")
print(f"Final Test shape:  {X_test.shape}\n")


Final Train shape: (8800000, 28)
Final Valid shape: (1100000, 28)
Final Test shape:  (1100000, 28)



### Model Training

In [5]:
MAX_EPOCHS = 500

In [6]:
# Hyperparameters from original paper (TabNet-S model).
# Note: momentum is 1 - 0.6 (paper m_B) to align with PyTorch's BatchNorm API.
paper_config = {
    'n_d': 24,
    'n_a': 26,
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.4,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': dict(lr=0.02),
    'mask_type': 'sparsemax'
}

clf_vanilla = TabNetClassifier(
    **paper_config,
    clip_value=2.,
    scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_params=dict(T_max=MAX_EPOCHS, eta_min=1e-5),
    use_kan=False,
    seed=seed,
    verbose=25
)

In [7]:
# Hyperparameters from original paper (TabNet-S model).
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_vanilla.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=75,
    num_workers=os.cpu_count(),
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 0.61869 | valid_accuracy: 0.68552 |  0:01:23s
epoch 25 | loss: 0.47725 | valid_accuracy: 0.76624 |  0:36:02s
epoch 50 | loss: 0.46574 | valid_accuracy: 0.77292 |  1:11:10s
epoch 75 | loss: 0.46252 | valid_accuracy: 0.77477 |  1:46:13s
epoch 100| loss: 0.46251 | valid_accuracy: 0.77562 |  2:21:02s
epoch 125| loss: 0.45979 | valid_accuracy: 0.77488 |  2:55:40s
epoch 150| loss: 0.45895 | valid_accuracy: 0.77695 |  3:30:18s
epoch 175| loss: 0.4582  | valid_accuracy: 0.77709 |  4:05:35s
epoch 200| loss: 0.45769 | valid_accuracy: 0.77748 |  4:40:51s
epoch 225| loss: 0.45723 | valid_accuracy: 0.7777  |  5:15:18s
epoch 250| loss: 0.4568  | valid_accuracy: 0.77812 |  5:49:47s
epoch 275| loss: 0.45637 | valid_accuracy: 0.77846 |  6:24:16s
epoch 300| loss: 0.45609 | valid_accuracy: 0.77838 |  6:58:45s
epoch 325| loss: 0.4558  | valid_accuracy: 0.7783  |  7:33:11s
epoch 350| loss: 0.45551 | valid_accuracy: 0.77896 |  8:07:39s
epoch 375| loss: 0.45528 | valid_accuracy: 0.77927 |  8

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_vanilla.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 76,680


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_vanilla.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.77941


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/higgs_boson/tables'
MODELS_DIR = f'{BASE_DIR}/results/higgs_boson/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "Vanilla TabNet",
    "dataset": "Higgs Boson",
    "parameters": param_count,
    "scheduler": "CosineAnnealingLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_vanilla.best_epoch,
    "training_history": clf_vanilla.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/03_vanilla_baseline_cosine_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_vanilla.save_model(f'{MODELS_DIR}/03_vanilla_baseline_cosine_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/higgs_boson/tables/03_vanilla_baseline_cosine_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/higgs_boson/models/03_vanilla_baseline_cosine_lr_76k.zip
